In [ ]:
import config

import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import json
import time
import random

In [2]:
bts = 'BTS'
blackpink = 'BLACKPINK'
exo = 'EXO'
twice = 'TWICE'

# Scraping lyrics

In [3]:
genius_client_access_token = config.genius_client_access_token
genius_api = 'https://api.genius.com'
headers = { "Authorization" : f"Bearer {genius_client_access_token}" }

## Getting the artist ID

In [4]:
# gets the artist ID given an artist parameter 
def getArtistID(artist):
    # getting the website data 
    response = requests.get(
        f'{genius_api}/search',
        params={'q':artist},
        headers=headers
    )
    all_data = response.json()

    # finding the accurate ID
    for index, data in enumerate(all_data['response']['hits']):
        curr_name = (all_data['response']['hits'][index]['result']['primary_artist']['name']).upper()
        if artist == curr_name:
            id = all_data['response']['hits'][index]['result']['primary_artist']['id']
            return id
        
    return -1

In [5]:
bts_id = getArtistID('BTS')
print(bts_id) # 70113

blackpink_id = getArtistID('BLACKPINK')
print(blackpink_id) # 987404

exo_id = getArtistID('EXO')
print(exo_id) # 67399

twice_id = getArtistID('TWICE')
print(twice_id) # 211474

70113
987404
67399
211474


## Getting all songs & data

### Storing songs -> dict

In [ ]:
# gets all songs from an artist 
# returns a list of song data

def getAllSongs(artist_id):
    # basic parameters
    per_page = 50
    page = 1
    hasNextPage = True 
    songs_list = []

    # looping through all pages
    while hasNextPage:

        songs_req = requests.get(
            f'{genius_api}/artists/{artist_id}/songs?',
            params={'per_page':per_page, 'page':page},
            headers=headers
        )
        
        songs_json = songs_req.json()

        for song in songs_json['response']['songs']:
            songs_list.append({
                'id': song['id'],
                'title': song['title'],
                'release_date_components': song['release_date_components'],
                'url': song['url']
            })
        
        page += 1
        hasNextPage = songs_json['response']['next_page'] != None
        

    return songs_list


In [7]:
bts_songs = getAllSongs(bts_id)

blackpink_songs = getAllSongs(blackpink_id)

exo_songs = getAllSongs(exo_id)

twice_songs = getAllSongs(twice_id)

In [ ]:
print(len(bts_songs)) #375

print(len(blackpink_songs)) #121

print(len(exo_songs)) #360

print(len(twice_songs)) #382

376
124
361
383


### Writing songs dictionary data into json file

In [ ]:
# writing songs dictionary into json file
def writeSongs(fileName, songs):
    file = open(fileName, 'w') 
    json.dump(songs,file,indent=4)

In [ ]:
# files then moved to: raw_data/artist_songs_data

bts_file = 'bts_songs.json'
writeSongs(bts_file, bts_songs)

blackpink_file = 'blackpink_songs.json'
writeSongs(blackpink_file, blackpink_songs)

exo_file = 'exo_songs.json'
writeSongs(exo_file, exo_songs)

twice_file = 'twice_songs.json'
writeSongs(twice_file, twice_songs)

## Getting lyrics

### Writing lyrics into txt file 

In [ ]:
# function to write lyrics to a file

def writeLyrics(fileName, songs):
    # opening the file to write lyrics
    lyrics_file = open(fileName, 'w')

    # writing lyrics for each song
    for song in songs:
        website = requests.get(song['url'])
        soup = BeautifulSoup(website.text, 'lxml')
        lyrics = soup.find_all('div', class_ = 'Lyrics__Container-sc-a49d8432-1 fBKwZw')

        lyrics_file.write(f'SONG NAME [{song['title']}]\n')
        
        for div in lyrics: 
            for d in div:
                if d.text != '':
                    lyrics_file.write(f'{d.text}\n')
            lyrics_file.write('\n\n')



In [ ]:
# files then moved to: raw_data/lyrics

bts_file = 'bts_lyrics.txt'
writeLyrics(bts_file, bts_songs)

blackpink_file = 'blackpink_lyrics.txt'
writeLyrics(blackpink_file, blackpink_songs)

exo_file = 'exo_lyrics.txt'
writeLyrics(exo_file, exo_songs)

twice_file = 'twice_lyrics.txt'
writeLyrics(twice_file, twice_songs)

# Spotify

In [9]:
import base64
from requests import post 
import json 

spotify_client_id = config.spotify_client_id
spotify_client_secret = config.spotify_client_secret

spotify_api = 'https://api.spotify.com/v1'


## Getting token

In [10]:
# getting token -> used for future headers for getting any data 
# https://developer.spotify.com/documentation/web-api/tutorials/client-credentials-flow
# https://www.youtube.com/watch?v=WAmEZBEeNmg&ab_channel=AkamaiDeveloper
def getToken():
    auth_string = f'{spotify_client_id}:{spotify_client_secret}'
    encoded_auth_string = base64.b64encode(auth_string.encode("utf-8")).decode("utf-8")

    url = 'https://accounts.spotify.com/api/token'

    headers = {
        "Authorization" : "Basic " + encoded_auth_string,
        "Content-Type" : "application/x-www-form-urlencoded"
    }

    data = {"grant_type" : "client_credentials"}
    result = post(url, headers=headers, data=data)
    json_result = json.loads(result.content)
    token = json_result["access_token"]
    return token

def getAuthHeader(token):
    return { "Authorization" : f"Bearer {token}" }

spotify_token = getToken()
spotify_headers = getAuthHeader(spotify_token)


## Getting Spotify ID

In [ ]:
# uses a search request to find the Spotify ID of an artist 
def getArtistID_spotify(artist):
    # sending a search request to the Spotify API
    search_req = requests.get(
        f'{spotify_api}/search',
        params={
            'q' : artist,
            'type' : 'artist'
        },
        headers= spotify_headers
    )

    search_json = search_req.json()

    # iterating through the results until the correct ID is found 
    for item in search_json['artists']['items']:
        if item['name'].upper() == artist.upper():
            return item['id']
        
    return -1


In [12]:
bts_spotify_id = getArtistID_spotify('BTS')
print(bts_spotify_id)

blackpink_spotify_id = getArtistID_spotify('blackpink')
print(blackpink_spotify_id)

exo_spotify_id = getArtistID_spotify('exo')
print(exo_spotify_id)

twice_spotify_id = getArtistID_spotify('twice')
print(twice_spotify_id)

3Nrfpe0tUJi4K4DXYWgMUX
41MozSoPIsD1dJM0CLPjZF
3cjEqqelV9zb4BYE3qDQ4O
7n2Ycct7Beij7Dj7meI4X0


## Getting popularity 

In [163]:
# get artist popularity value from Spotify 

def getArtistPopularity(id):
    
    artist_req = requests.get(
        f'{spotify_api}/artists/{id}',
        # params={
        #     'id' : artist_id,
        # },
        headers= spotify_headers
    )

    artist_json = artist_req.json()

    return artist_json['popularity']



## Getting albums

In [ ]:
# getting all albums of an artist 

def getAlbums(id):
    # basic parameters 
    albums = []
    hasNext = True
    offset = 0
    limit = 2

    # iterating through all pages 
    while hasNext:

    # sending a request to get the albums of the artist
        albums_req = requests.get(
            f'{spotify_api}/artists/{id}/albums',
            params= {
                # 'include_groups' : 'album,single',
                'include_groups' : 'album',
                'limit' : limit,
                'offset' : offset
            },
            headers= spotify_headers
        )

        albums_json = albums_req.json()

        # iterating through the results to add the information needed for analysis 
        for album in albums_json['items']:
            albums.append(
                {
                    'name' : album['name'],
                    'release_date' : album['release_date'],
                    'release_date_precision' : album['release_date_precision'],
                    'num_available_markets' : len(album['available_markets']),
                    'available_markets' : album['available_markets']
                }
            )

        offset += limit
        hasNext = albums_json['next'] != None
    
    return albums


In [165]:
bts_albums = getAlbums(bts_spotify_id)

blackpink_albums = getAlbums(blackpink_spotify_id)

exo_albums = getAlbums(exo_spotify_id)

twice_albums = getAlbums(twice_spotify_id)


## Getting top tracks 

In [166]:
# getting top tracks of an artist 
def getTopTracks(id):

    top_tracks = []

    top_req = requests.get(
        f'{spotify_api}/artists/{id}/top-tracks',
        headers= spotify_headers
    )

    top_json = top_req.json()

    for track in top_json['tracks']:
        top_tracks.append(
            {
                'album' : track['album']['name'],
                'name' : track['name'],
                'popularity' : track['popularity'],
                'release_date' : track['album']['release_date'],
                'release_date_precision' : track['album']['release_date_precision'],
                'num_available_markets' : len(track['available_markets']),
                'available_markets' : track['available_markets']
            }
        )
        
    return top_tracks



In [167]:
bts_top = getTopTracks(bts_spotify_id)

blackpink_top = getTopTracks(blackpink_spotify_id)

exo_top = getTopTracks(exo_spotify_id)

twice_top = getTopTracks(twice_spotify_id)


## Finalizing individual artist data structure

In [168]:
bts_data = {
    'name' : bts,
    'id' : bts_spotify_id,
    'popularity' : getArtistPopularity(bts_spotify_id),
    'albums' : bts_albums,
    'top_tracks' : bts_top
}

blackpink_data = {
    'name' : blackpink,
    'id' : blackpink_spotify_id,
    'popularity' : getArtistPopularity(blackpink_spotify_id),
    'albums' : blackpink_albums,
    'top_tracks' : blackpink_top
}

exo_data = {
    'name' : exo,
    'id' : exo_spotify_id,
    'popularity' : getArtistPopularity(exo_spotify_id),
    'albums' : exo_albums,
    'top_tracks' : exo_top
}

twice_data = {
    'name' : twice,
    'id' : twice_spotify_id,
    'popularity' : getArtistPopularity(twice_spotify_id),
    'albums' : twice_albums,
    'top_tracks' : twice_top
}


# df = pd.DataFrame.from_dict(bts_data, orient='index')
# print(df)

# df = pd.DataFrame(bts_data['top_tracks'])
# df

## Writing artist data files

In [ ]:
# files then moved to: raw_data/artist_data

bts_file = 'bts_data.json'
writeSongs(bts_file, bts_data)

blackpink_file = 'blackpink_data.json'
writeSongs(blackpink_file, blackpink_data)

exo_file = 'exo_data.json'
writeSongs(exo_file, exo_data)

twice_file = 'twice_data.json'
writeSongs(twice_file, twice_data)

# Billboard 

In [171]:
billboard_website = 'https://www.billboard.com/artist'

In [357]:
def cleanTitle(text):
    text.replace('\n','\t')
    if '\t' in text:
        text = text[0:text.index('\t')]
    return text

In [ ]:
# getting Billboard Global data for an artist

def getBillboardGlobal(artist):
    # setting up the Billboard Global data structure
    billboard_data = {}
    billboard_data['name'] = artist
    billboard_data['billboard_data'] = []

    # sending a request and using BeautifulSoupto get the Billboard Global data
    website = requests.get(f'{billboard_website}/{artist}/chart-history/glo')    
    soup = BeautifulSoup(website.text, 'lxml')

    contents = soup.find_all('div',class_='o-chart-results-list-row // lrv-u-flex lrv-u-flex-direction-column@mobile-max lrv-u-background-color-white u-height-74@tablet u-padding-tb-0.375@mobile-max u-border-radius-a-6')

    song = contents[0].find('div',class_='o-chart-results-list__item // lrv-u-flex lrv-u-flex-direction-column lrv-u-flex-grow-1 lrv-u-justify-content-center lrv-u-padding-lr-075@mobile-max u-padding-l-1.50@tablet u-padding-r-2.875@tablet u-height-52@mobile-max')
    # print(song.text.strip())

    # iterating through the data to add the necessary information
    for index, c in enumerate(contents):
        billboard_data['billboard_data'].append({})
        song = c.find('div',class_='o-chart-results-list__item // lrv-u-flex lrv-u-flex-direction-column lrv-u-flex-grow-1 lrv-u-justify-content-center lrv-u-padding-lr-075@mobile-max u-padding-l-1.50@tablet u-padding-r-2.875@tablet u-height-52@mobile-max')
        song_name = cleanTitle(song.text.strip()[0:-len(artist)].strip())
        # print(song_name)
        billboard_data['billboard_data'][index]['song_name'] = song_name

        song_data = c.find('div',class_='lrv-u-flex lrv-u-height-100p u-height-40@mobile-max lrv-u-justify-content-end')

        debut_date = song_data.find('div',class_='o-chart-results-list__item // u-width-120 u-width-82@mobile-max lrv-u-flex lrv-u-align-items-center lrv-u-justify-content-center lrv-u-border-r-2 lrv-u-border-color-grey-lightest')
        # print(f'Debut Date: {debut_date.text.strip()}')
        billboard_data['billboard_data'][index]['debut_date'] = debut_date.text.strip()

        peak_pos = song_data.find('div',class_='o-chart-results-list__item // u-width-70@tablet u-width-55@mobile-max lrv-u-flex lrv-u-flex-direction-column lrv-u-align-items-center lrv-u-justify-content-center u-background-color-white-064@mobile-max lrv-u-border-r-2 lrv-u-border-color-grey-lightest').text.strip()
        peak_pos = peak_pos[0:-len('12 WKS')].strip()
        # print(f'Peak Position: {peak_pos}')
        billboard_data['billboard_data'][index]['peak_position'] = int(peak_pos)

        peak_date = debut_date.find_next('div',class_='o-chart-results-list__item // u-width-120 u-width-82@mobile-max lrv-u-flex lrv-u-align-items-center lrv-u-justify-content-center lrv-u-border-r-2 lrv-u-border-color-grey-lightest').text.strip()
        # print(f'Peak Date: {peak_date}')
        billboard_data['billboard_data'][index]['peak_date'] = peak_date

        weeks = song_data.find('div',class_='o-chart-results-list__item // u-width-70@tablet u-width-55@mobile-max lrv-u-flex lrv-u-align-items-center lrv-u-justify-content-center u-background-color-white-064@mobile-max lrv-u-border-r-2 lrv-u-border-color-grey-lightest').text.strip()
        # print(f'Weeks on Chart: {weeks}')
        billboard_data['billboard_data'][index]['weeks_on_chart'] = int(weeks)

        # print()

    # to avoid overwhelming the server with requests
    print('Waiting')
    time.sleep(random.randint(3,6))
    print('done')

    return billboard_data



In [420]:
def writeBillboard(artist, data):
    file = open(f'{artist.lower()}_billboard.json','w')
    json.dump(data,file,indent=4)

In [ ]:
# files then moved to: raw_data/billboard_data/
writeBillboard(bts,getBillboardGlobal(bts))
writeBillboard(blackpink,getBillboardGlobal(blackpink))
writeBillboard(exo,getBillboardGlobal(exo))
writeBillboard(twice,getBillboardGlobal(twice))

Waiting
done
Waiting
done
Waiting
done
Waiting
done
